In [1]:
# 경고 메시지 무시
import warnings
warnings.filterwarnings(action='ignore') 

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import csv
import folium
import datetime
import seaborn as sns
import scipy as sp
import statsmodels.formula.api as smf
import networkx as nx
import missingno as msno
import os
import sys
import urllib.request
import time
import json
import plotly.express as px
import re
import sklearn.metrics as metrics

from sklearn.decomposition import PCA
from sklearn.datasets import load_iris, load_wine, load_breast_cancer, load_diabetes
from folium.plugins import HeatMap 
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from dateutil.relativedelta import relativedelta
from sklearn.cluster import KMeans    
from yellowbrick.cluster import KElbowVisualizer
from scipy.cluster.hierarchy import dendrogram, linkage
from mpl_toolkits.mplot3d import Axes3D
from operator import itemgetter
from PIL import Image
from collections import Counter
from wordcloud import WordCloud
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import r2_score, ConfusionMatrixDisplay, confusion_matrix, accuracy_score, silhouette_score
from sklearn.neighbors import KNeighborsClassifier

# plt.rc('font', family='malgun gothic')
# plt.rcParams['axes.unicode_minus']=False  # '- 표시
plt.rc('font',family='D2CodingLigature Nerd Font')

## 1. 데이터 읽기

In [5]:
CB = pd.read_csv( '../../data/CoffeeBean.csv', encoding = 'CP949', index_col = 0, header = 0)
CB.head()

,store,address,phone
0,차병원점,서울시 강남구 논현로 566 강남차병원1층,02-538-7615
1,강남대로점,서울시 서초구 강남대로 369 1층,02-588-5778
2,청담에스점,"서울시 강남구 압구정로 461 네이처포엠빌딩B108,109호",02-548-6052
3,신사점,서울시 강남구 도산대로 126,02-548-2741
4,역삼점,"서울시 강남구 논현로 512 지상1,2층",02-569-8051


## 2. 데이터 전처리

In [6]:
addr = []
for address in CB.address:
    addr.append(str(address).split())
    
addr

[['서울시', '강남구', '논현로', '566', '강남차병원1층'],
 ['서울시', '서초구', '강남대로', '369', '1층'],
 ['서울시', '강남구', '압구정로', '461', '네이처포엠빌딩B108,109호'],
 ['서울시', '강남구', '도산대로', '126'],
 ['서울시', '강남구', '논현로', '512', '지상1,2층'],
 ['서울시', '서초구', '강남대로', '213', '24호', '지하1층'],
 ['서울시', '강남구', '삼성로', '716', 'LEE76빌딩2층'],
 ['서울', '서초구', '반포동', '736-17', 'P빌딩', '2층'],
 ['서울시', '강남구', '언주로', '30길', '10,112', '현대비젼21', '112호'],
 ['서울시', '강남구', '선릉로', '749', '1,2층'],
 ['서울시', '강남구', '도산대로49길', '13', '1층', '17,18호'],
 ['서울시', '서초구', '강남대로', '51길', '1', '511', 'TOWER', '1층'],
 ['서울시', '강남구', '논현', '231-13호', '팍스타워지하1층'],
 ['서울', '강남구', '테헤란로87길', '46', '지하', '2층'],
 ['서울시', '강남구', '영동대로', '511', '트레이드타워', '지하1층'],
 ['서울시', '강남구', '영동대로', '607', '1,2층'],
 ['서울시', '송파구', '석촌호수로', '118', '1층'],
 ['서울시', '서초구', '서초동', '1685-8호', '101~2호,113~4호,121호'],
 ['서울시', '강남구', '논현로', '717', '1층'],
 ['서울시', '서초구', '서초대로74길', '11', '지하2층'],
 ['서울시', '서초구', '서초대로74길', '4', '삼성생명보험서초타워내지하1층'],
 ['서울특별시', '서초구', '방배중앙로', '187'],
 ['서울시',

In [7]:
addr2 = []
for i in range(len(addr)):    
    if addr[i][0] == "서울": addr[i][0] = "서울특별시"
    elif addr[i][0] == "서울시": addr[i][0] = "서울특별시"
    elif addr[i][0] == "부산시": addr[i][0] = "부산광역시"
    elif addr[i][0] == "부산": addr[i][0] = "부산광역시"
    elif addr[i][0] == "인천": addr[i][0] = "인천광역시"
    elif addr[i][0] == "인천시": addr[i][0] = "인천광역시"
    elif addr[i][0] == "광주": addr[i][0] = "광주광역시"
    elif addr[i][0] == "대전시": addr[i][0] = "대전광역시"
    elif addr[i][0] == "울산시": addr[i][0] = "울산광역시"
    elif addr[i][0] == "세종시": addr[i][0] = "세종특별자치시"
    elif addr[i][0] == "경기": addr[i][0] = "경기도"
    elif addr[i][0] == "충북": addr[i][0] = "충청북도"
    elif addr[i][0] == "충남": addr[i][0] = "충청남도"
    elif addr[i][0] == "전북": addr[i][0] = "전라북도"
    elif addr[i][0] == "전남": addr[i][0] = "전라남도"
    elif addr[i][0] == "경북": addr[i][0] = "경상북도"
    elif addr[i][0] == "경남": addr[i][0] = "경상남도"
    elif addr[i][0] == "제주": addr[i][0] = "제주특별자치도"
    elif addr[i][0] == "제주도": addr[i][0] = "제주특별자치도"
    elif addr[i][0] == "제주시": addr[i][0] = "제주특별자치도"
    addr2.append(' '.join(addr[i]))    

addr2 

['서울특별시 강남구 논현로 566 강남차병원1층',
 '서울특별시 서초구 강남대로 369 1층',
 '서울특별시 강남구 압구정로 461 네이처포엠빌딩B108,109호',
 '서울특별시 강남구 도산대로 126',
 '서울특별시 강남구 논현로 512 지상1,2층',
 '서울특별시 서초구 강남대로 213 24호 지하1층',
 '서울특별시 강남구 삼성로 716 LEE76빌딩2층',
 '서울특별시 서초구 반포동 736-17 P빌딩 2층',
 '서울특별시 강남구 언주로 30길 10,112 현대비젼21 112호',
 '서울특별시 강남구 선릉로 749 1,2층',
 '서울특별시 강남구 도산대로49길 13 1층 17,18호',
 '서울특별시 서초구 강남대로 51길 1 511 TOWER 1층',
 '서울특별시 강남구 논현 231-13호 팍스타워지하1층',
 '서울특별시 강남구 테헤란로87길 46 지하 2층',
 '서울특별시 강남구 영동대로 511 트레이드타워 지하1층',
 '서울특별시 강남구 영동대로 607 1,2층',
 '서울특별시 송파구 석촌호수로 118 1층',
 '서울특별시 서초구 서초동 1685-8호 101~2호,113~4호,121호',
 '서울특별시 강남구 논현로 717 1층',
 '서울특별시 서초구 서초대로74길 11 지하2층',
 '서울특별시 서초구 서초대로74길 4 삼성생명보험서초타워내지하1층',
 '서울특별시 서초구 방배중앙로 187',
 '서울특별시 강남구 테헤란로 20길 10 쓰리엠타워1층',
 '서울특별시 강남구 테헤란로 70길 12 1층',
 '서울특별시 강남구 봉은사로 628 엘슨빌딩1층',
 '서울특별시 송파구 오금로 11길 7',
 '서울특별시 강남구 논현로 38길 42 418호',
 '서울특별시 서초구 서초중앙로 43 로얄타워1층',
 '서울특별시 강남구 테헤란로4길 28 826-28호 1층',
 '서울특별시 강남구 테헤란로87길 36 공항타워 1층',
 '서울특별시 강남구 도산대로 67길 7,파크빌딩 1층',
 '서울특별시 종로구 서린동 13

In [8]:
addr2 = pd.DataFrame(addr2, columns = ['address2'])
addr2.head(3)

,address2
0,서울특별시 강남구 논현로 566 강남차병원1층
1,서울특별시 서초구 강남대로 369 1층
2,"서울특별시 강남구 압구정로 461 네이처포엠빌딩B108,109호"


In [9]:
CB2 = pd.concat([CB, addr2], axis = 1 )
CB2.head()

,store,address,phone,address2
0,차병원점,서울시 강남구 논현로 566 강남차병원1층,02-538-7615,서울특별시 강남구 논현로 566 강남차병원1층
1,강남대로점,서울시 서초구 강남대로 369 1층,02-588-5778,서울특별시 서초구 강남대로 369 1층
2,청담에스점,"서울시 강남구 압구정로 461 네이처포엠빌딩B108,109호",02-548-6052,"서울특별시 강남구 압구정로 461 네이처포엠빌딩B108,109호"
3,신사점,서울시 강남구 도산대로 126,02-548-2741,서울특별시 강남구 도산대로 126
4,역삼점,"서울시 강남구 논현로 512 지상1,2층",02-569-8051,"서울특별시 강남구 논현로 512 지상1,2층"


* folium은 파이썬으로 ‘인터랙티브 지도’를 만드는 라이브러리임.
* 웹 지도(줌, 클릭, 마커)를 파이썬 코드로 만들 수 있음.
* 마커 색(color): red, blue, green, purple, orange, darkred, cadetblue, black, gray
* 마커 모양(icon): info-sign(기본값), star, home, coffee, shopping-cart, user, ok, remove, flag, map-marker
* 목록에 없는거 사용시 color는 blue, icon은 info-sign 표기

In [10]:
map_osm = folium.Map(location = [37.559978, 126.975291], zoom_start = 16)

folium.Marker(
    location=[37.559978, 126.975291],
    tooltip='중심 위치',
    icon=folium.Icon(color='red', icon='star')
).add_to(map_osm)

map_osm.save('../../data/map.html')

## 3. 지오서비스웹에서 위도, 경도값 알아내기

In [ ]:
# 정제된 파일 저장

CB2.to_csv("../../data/CoffeeBean_2.csv", encoding='CP949', index=False)

### 변환 데이터가 다운로드 되었다고 가정
* ### https://www.vworld.kr/dev/v3dv_geocoderguide2_s001.do 를 이용하면 된다

In [14]:
CB_geoData = pd.read_csv('../../data/CB_geoResult.csv', encoding = 'utf8', engine = 'python')
CB_geoData.head(2)

,store,address,phone,address2,_GC_TYPE,_CLEANADDR,_X,_Y
0,차병원점,서울시 강남구 논현로 566 강남차병원1층,02-538-7615,서울특별시 강남구 논현로 566 강남차병원1층,정,서울특별시 강남구 논현로 566 (역삼동),127.034738,37.50704
1,역삼점,"서울시 강남구 논현로 512 지상1,2층",02-569-8051,"서울특별시 강남구 논현로 512 지상1,2층",정,서울특별시 강남구 논현로 512 (역삼동),127.036521,37.50235


In [20]:
# 1. 지도 객체를 먼저 생성해야 합니다 (서울 시청 기준 예시)
map_CB = folium.Map(location=[37.5665, 126.9780], zoom_start=12)

for i, store in CB_geoData.iterrows():
    folium.Marker( location = [store['_Y'], store['_X']], tooltip = store['store'], 
                  icon = folium.Icon(color = 'red', icon = 'star')).add_to(map_CB)
                  # icon = folium.Icon()).add_to(map_CB) #default test
map_CB.save('../../data/map_CB.html')